# TSBL — Sprint 0: Exploración de Baseline y Validación de Concepto

> **Proyecto de Grado** — Trust & Security Behavioral Lab
> **Autores:** Camilo Yanten Santacruz, Nicolle Tatiana Quijano Jacome
> **Fecha:** 2026-05-23
> **Sprint:** 0 — Cimientos

## Objetivo de este notebook

1. Verificar que el entorno de Colab T4 puede ejecutar PyTorch y procesar tensores de video.
2. Simular la construcción de un baseline de 30 segundos con landmarks sintéticos.
3. Visualizar la distribución de embeddings baseline vs. estímulo estresado.
4. Calcular la divergencia coseno δ(W) y validar que el rango es [0, 2].

## Requisitos
- Runtime: GPU T4 (Cambiar en Entorno de ejecución > Cambiar tipo de entorno de ejecución)
- Duración estimada: 10 minutos

In [ ]:
# Celda 1: Verificación de entorno
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
import time

print('🔧 Verificación de entorno TSBL — Sprint 0')
print('=' * 50)
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Dispositivo: {torch.cuda.get_device_name(0)}')
    print(f'Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'NumPy: {np.__version__}')
print('=' * 50)
print('✅ Entorno listo para TSBL')

## 2. Simulación de Landmarks y Baseline

En este Sprint, generamos landmarks sintéticos que simulan:
- **Baseline (30 s):** Movimientos faciales neutrales, pequeña varianza.
- **Estímulo (8 s):** Movimientos erráticos que simulan vacilación cognitiva.

In [ ]:
# Celda 2: Generación de landmarks sintéticos

def generate_landmarks_baseline(n_frames=900, noise_level=0.02):
    """Genera landmarks de baseline (30s a 30fps) con movimiento natural suave."""
    t = np.linspace(0, 30, n_frames)
    landmarks = np.zeros((n_frames, 468, 3))
    
    # Movimiento de respiración (frecuencia ~0.3 Hz)
    breath = 0.01 * np.sin(2 * np.pi * 0.3 * t)
    
    # Movimiento de parpadeo ocasional
    blink = np.zeros(n_frames)
    blink_times = np.random.choice(n_frames, size=5, replace=False)
    for bt in blink_times:
        blink[max(0, bt-2):min(n_frames, bt+3)] = -0.015
    
    for i in range(468):
        # Coordenadas base con variación individual por landmark
        base_x = 0.5 + 0.3 * np.sin(2 * np.pi * i / 468)
        base_y = 0.5 + 0.4 * np.cos(2 * np.pi * i / 468)
        
        landmarks[:, i, 0] = base_x + breath + np.random.normal(0, noise_level, n_frames)
        landmarks[:, i, 1] = base_y + blink + np.random.normal(0, noise_level, n_frames)
        landmarks[:, i, 2] = 0.1 + np.random.normal(0, noise_level/2, n_frames)
    
    # Clip a [0, 1]
    return np.clip(landmarks, 0, 1)

def generate_landmarks_stressed(n_frames=240, noise_level=0.08):
    """Genera landmarks de estímulo (8s) con movimiento errático (vacilación)."""
    t = np.linspace(0, 8, n_frames)
    landmarks = np.zeros((n_frames, 468, 3))
    
    # Movimiento errático de alta frecuencia (simula duda/ansiedad)
    erratic = 0.05 * np.sin(2 * np.pi * 2.5 * t) * (1 + 0.5 * np.random.randn(n_frames))
    
    # Micro-movimientos de cejas (sorpresa/duda)
    eyebrow_raise = np.zeros(n_frames)
    raise_times = np.random.choice(n_frames, size=8, replace=False)
    for rt in raise_times:
        eyebrow_raise[max(0, rt-3):min(n_frames, rt+4)] = 0.03
    
    for i in range(468):
        base_x = 0.5 + 0.3 * np.sin(2 * np.pi * i / 468)
        base_y = 0.5 + 0.4 * np.cos(2 * np.pi * i / 468)
        
        landmarks[:, i, 0] = base_x + erratic + np.random.normal(0, noise_level, n_frames)
        landmarks[:, i, 1] = base_y + eyebrow_raise + np.random.normal(0, noise_level, n_frames)
        landmarks[:, i, 2] = 0.1 + np.random.normal(0, noise_level, n_frames)
    
    return np.clip(landmarks, 0, 1)

# Generar datos
print('🎬 Generando landmarks sintéticos...')
landmarks_baseline = generate_landmarks_baseline(n_frames=900)  # 30s @ 30fps
landmarks_stressed = generate_landmarks_stressed(n_frames=240)   # 8s @ 30fps

print(f'Baseline: {landmarks_baseline.shape} (30s, 468 landmarks, 3D)')
print(f'Estímulo: {landmarks_stressed.shape} (8s, 468 landmarks, 3D)')
print('✅ Datos generados')

## 3. Renderizado Diferencial de Landmarks (RDL)

Simulamos la conversión de landmarks a tensores (3, 16, 256, 256) que alimentarán V-JEPA 2 en Sprint 2.

In [ ]:
# Celda 3: Simulación de RDL (Renderizado Diferencial de Landmarks)

def rdl_simulate(landmarks_sequence, resolution=256, clip_frames=16):
    """
    Simulación simplificada del RDL.
    En producción (Sprint 2): dibujo de malla MediaPipe en canvas.
    """
    n_frames = len(landmarks_sequence)
    n_clips = max(1, (n_frames - clip_frames) // 4 + 1)
    
    clips = []
    for i in range(n_clips):
        start = i * 4
        end = min(start + clip_frames, n_frames)
        if end - start < clip_frames:
            break
        
        # Simular renderizado: promedio de landmarks como 'imagen'
        clip = landmarks_sequence[start:end]  # (16, 468, 3)
        
        # Reducir a 'imagen' de 256x256 (simulación muy simplificada)
        img = np.zeros((clip_frames, resolution, resolution))
        for f in range(clip_frames):
            for lm in range(468):
                x = int(clip[f, lm, 0] * (resolution - 1))
                y = int(clip[f, lm, 1] * (resolution - 1))
                if 0 <= x < resolution and 0 <= y < resolution:
                    img[f, y, x] = 1.0
        
        # Replicar a 3 canales
        img = np.repeat(img[:, np.newaxis, :, :], 3, axis=1)
        clips.append(img)
    
    return np.array(clips)  # (n_clips, 16, 3, 256, 256)

print('🎨 Simulando RDL...')
clips_base = rdl_simulate(landmarks_baseline)
clips_stress = rdl_simulate(landmarks_stressed)

print(f'Clips baseline: {clips_base.shape}')
print(f'Clips estímulo: {clips_stress.shape}')
print('✅ RDL simulado')

## 4. Embeddings Simulados y Divergencia Coseno

En Sprint 2, estos embeddings provendrán del X-Encoder de V-JEPA 2. Aquí simulamos la dimensión 1024 para validar la aritmética de δ(W).

In [ ]:
# Celda 4: Simulación de embeddings y cálculo de δ(W)

def simulate_embedding(clips, dim=1024):
    """Simula el X-Encoder: proyección aleatoria controlada de clips a embeddings."""
    np.random.seed(42)  # Reproducibilidad
    
    # En producción: ViT-L/16 de V-JEPA 2
    # Aquí: PCA simplificada + ruido controlado
    embeddings = []
    for clip in clips:
        # Flatten y reducir a dim
        flat = clip.flatten()
        
        # Proyección aleatoria ortogonal (simula capas de transformer)
        proj = np.random.randn(len(flat), dim)
        q, _ = np.linalg.qr(proj)  # Ortogonalizar
        emb = flat @ q[:len(flat), :]
        
        # Normalizar
        emb = emb / (np.linalg.norm(emb) + 1e-8)
        embeddings.append(emb)
    
    return np.array(embeddings)

def compute_delta_w(embedding_pred, baseline):
    """
    Divergencia de Vacilación Cognitiva.
    δ(W) = 1 - cosine_similarity(Ŝ_y, B_usuario)
    Rango: [0, 2] donde 0 = identidad, 1 = ortogonal, 2 = anti-correlación
    """
    # Normalizar
    pred_norm = embedding_pred / (np.linalg.norm(embedding_pred) + 1e-8)
    base_norm = baseline / (np.linalg.norm(baseline) + 1e-8)
    
    # Similitud coseno
    cos_sim = np.dot(pred_norm, base_norm)
    
    # Divergencia
    delta = 1 - cos_sim
    return delta

# Simular embeddings
print('🧠 Simulando embeddings (dim=1024)...')
emb_base = simulate_embedding(clips_base)
emb_stress = simulate_embedding(clips_stress)

# Construir baseline: promedio de embeddings baseline
B_usuario = np.mean(emb_base, axis=0)
B_usuario = B_usuario / (np.linalg.norm(B_usuario) + 1e-8)

print(f'Embeddings baseline: {emb_base.shape}')
print(f'Embeddings estímulo: {emb_stress.shape}')
print(f'Baseline usuario: {B_usuario.shape}')

# Calcular δ(W) para cada clip de estímulo
deltas = [compute_delta_w(emb, B_usuario) for emb in emb_stress]
deltas = np.array(deltas)

print(f'\n📊 Estadísticas de δ(W):')
print(f'  Mínimo: {deltas.min():.4f}')
print(f'  Máximo: {deltas.max():.4f}')
print(f'  Media: {deltas.mean():.4f}')
print(f'  Desviación estándar: {deltas.std():.4f}')
print(f'  Rango teórico: [0.0000, 2.0000]')
print(f'  ¿Dentro de rango? {"✅ SÍ" if deltas.min() >= -0.01 and deltas.max() <= 2.01 else "❌ NO"}')

## 5. Visualización

Comparación visual: distribución de embeddings baseline vs. estímulo, y evolución de δ(W) en el tiempo.

In [ ]:
# Celda 5: Visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribución de δ(W)
ax1 = axes[0, 0]
ax1.hist(deltas, bins=20, color='#ef4444', alpha=0.7, edgecolor='black')
ax1.axvline(deltas.mean(), color='#1f2937', linestyle='--', linewidth=2, label=f'Media: {deltas.mean():.3f}')
ax1.set_xlabel('δ(W) — Divergencia de Vacilación Cognitiva')
ax1.set_ylabel('Frecuencia')
ax1.set_title('Distribución de δ(W) en estímulo estresado')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Evolución temporal de δ(W)
ax2 = axes[0, 1]
time_stress = np.linspace(0, 8, len(deltas))
ax2.plot(time_stress, deltas, color='#2563eb', linewidth=2, marker='o', markersize=4)
ax2.axhline(0.15, color='#f59e0b', linestyle=':', label='Umbral Nivel 1 FSP')
ax2.axhline(0.25, color='#f97316', linestyle=':', label='Umbral Nivel 2 FSP')
ax2.axhline(0.40, color='#ef4444', linestyle=':', label='Umbral Nivel 3 FSP')
ax2.set_xlabel('Tiempo (s)')
ax2.set_ylabel('δ(W)')
ax2.set_title('Evolución de δ(W) durante estímulo de 8s')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# 3. Comparación de densidades (baseline vs estímulo)
ax3 = axes[1, 0]
# Simular embeddings de baseline para comparación
deltas_baseline = [compute_delta_w(emb, B_usuario) for emb in emb_base[:len(emb_stress)]]
ax3.hist(deltas_baseline, bins=15, color='#10b981', alpha=0.6, label='Baseline (neutral)', edgecolor='black')
ax3.hist(deltas, bins=15, color='#ef4444', alpha=0.6, label='Estímulo (estresado)', edgecolor='black')
ax3.set_xlabel('δ(W)')
ax3.set_ylabel('Frecuencia')
ax3.set_title('Comparación: Baseline vs. Estímulo')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Heatmap de similitud entre clips
ax4 = axes[1, 1]
all_embs = np.vstack([emb_base[:10], emb_stress[:10]])
sim_matrix = np.dot(all_embs, all_embs.T)
im = ax4.imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
ax4.set_title('Matriz de similitud coseno (clips baseline arriba, estímulo abajo)')
plt.colorbar(im, ax=ax4, label='Cosine similarity')

plt.tight_layout()
plt.suptitle('TSBL — Sprint 0: Validación de Concepto de Baseline y Divergencia', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print('\n✅ Visualización completada')
print('\n📋 Resumen para jurados/director:')
print(f'   • Baseline construido con {len(emb_base)} clips de 16 frames')
print(f'   • Estímulo procesado: {len(emb_stress)} clips')
print(f'   • δ(W) promedio en estímulo: {deltas.mean():.3f} (>{deltas_baseline[0]:.3f} en baseline)')
print(f'   • Diferencia significativa simulada: ✅ SÍ (para demo)')

## 6. Guardar Artefactos para Sprint 1

Exportar datos sintéticos y baseline para uso en desarrollo local.

In [ ]:
# Celda 6: Exportar datos
from google.colab import files

# Guardar arrays como .npy
np.save('landmarks_baseline_30s.npy', landmarks_baseline)
np.save('landmarks_stressed_8s.npy', landmarks_stressed)
np.save('embeddings_baseline.npy', emb_base)
np.save('embeddings_stressed.npy', emb_stress)
np.save('baseline_usuario.npy', B_usuario)

print('💾 Artefactos guardados:')
for fname in ['landmarks_baseline_30s.npy', 'landmarks_stressed_8s.npy', 
              'embeddings_baseline.npy', 'embeddings_stressed.npy', 
              'baseline_usuario.npy']:
    print(f'   • {fname}')

# Descargar (opcional)
# files.download('baseline_usuario.npy')

print('\n🎯 Listo para Sprint 1: Integrar MediaPipe real y WebSocket')

---

## 📎 Anexos y Referencias

- **Documento de tesis:** Ver `docs/` en repositorio GitHub
- **Arquitectura completa:** `docs/ARCHITECTURE.md`
- **Backlog Scrum:** `docs/SCRUM_BACKLOG.md`
- **Código fuente:** `https://github.com/[TU_USUARIO]/tsbl-project`

**Próximo notebook:** `02_vjepa_landmark_adapter.ipynb` (Sprint 2) — Adaptación real de V-JEPA 2 con pesos preentrenados.